In [2]:
import re
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

import nltk
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
nltk.download('stopwords', quiet=True)
from nltk.stem.porter import PorterStemmer
from nltk.corpus import stopwords as nltk_stopwords

In [3]:
SRC = "finalGagaData.csv"

df = pd.read_csv(SRC)
df = df.drop(columns=[c for c in ["Unnamed: 0", "index"] if c in df.columns])
df = df.reset_index(drop=True)
df.index.name = "song_id"

album_order = (
    df.groupby("album")["release_date"]
      .min()
      .sort_values(kind="stable")
      .reset_index()
)
album_order["album_id"] = range(len(album_order))
album_id_map = dict(zip(album_order["album"], album_order["album_id"]))

df["album_id"] = df["album"].map(album_id_map)

### LIB

In [4]:
lib = df[[
    "album_id",
    "track_name",
    "album",
    "track_number",
    "release_date",
    "duration_sec",
    "explicit",
    "artists",
    "streams",
]].copy()
lib["n_tokens"] = df["lyrics"].fillna("").str.split().map(len)
lib.to_csv("LIB.csv", index=True)

### CORPUS

In [5]:
TOKEN_RE = re.compile(r"[a-z0-9']+")

rows = []
for song_id, row in df.iterrows():
    album_id = row["album_id"]
    text = str(row["lyrics"]).lower()
    for token_id, match in enumerate(TOKEN_RE.finditer(text)):
        rows.append((album_id, song_id, token_id, match.group()))

corpus = pd.DataFrame(rows, columns=["album_id", "song_id", "token_id", "term_str"])
corpus = corpus.set_index(["album_id", "song_id", "token_id"])
corpus.to_csv("CORPUS.csv", index=True)

### VOCAB

In [6]:
cflat = corpus.reset_index()

term_count = cflat.groupby("term_str").size().rename("n")
doc_count  = cflat.groupby("term_str")["song_id"].nunique().rename("df")

N_docs   = df.shape[0]
N_tokens = int(term_count.sum())

vocab = pd.concat([term_count, doc_count], axis=1)
vocab = vocab.sort_values("n", ascending=False).reset_index()
vocab.index.name = "term_id"

vocab["p"]     = vocab["n"] / N_tokens
vocab["i"]     = -np.log2(vocab["p"])
vocab["idf"]   = np.log2(N_docs / vocab["df"])
vocab["dfidf"] = vocab["df"] * vocab["idf"]

ps = PorterStemmer()
vocab["porter_stem"] = vocab["term_str"].map(ps.stem)


pos_counts = defaultdict(Counter)
for song_id, group in cflat.groupby("song_id", sort=True):
    tokens = group["term_str"].tolist()
    for term, tag in nltk.pos_tag(tokens):
        pos_counts[term][tag] += 1

def _max_pos(term: str) -> str:
    counter = pos_counts.get(term)
    return counter.most_common(1)[0][0] if counter else "NN"

vocab["max_pos"] = vocab["term_str"].map(_max_pos)

PENN_TO_GROUP = {
    "NN": "N", "NNS": "N", "NNP": "N", "NNPS": "N",
    "VB": "V", "VBD": "V", "VBG": "V", "VBN": "V", "VBP": "V", "VBZ": "V",
    "MD": "V", "RP": "V",
    "JJ": "J", "JJR": "J", "JJS": "J",
    "RB": "R", "RBR": "R", "RBS": "R", "WRB": "R",
    "PRP": "P", "PRP$": "P", "WP": "P", "WP$": "P",
    "DT": "D", "PDT": "D", "WDT": "D",
    "IN": "I", "TO": "I",
    "CC": "C",
    "CD": "#",
    "UH": "U",
    "EX": "X", "POS": "X", "FW": "X", "LS": "X", "SYM": "X",
}
vocab["max_pos_group"] = vocab["max_pos"].map(lambda t: PENN_TO_GROUP.get(t, "X"))

STOPWORDS = set(nltk_stopwords.words("english"))
vocab["stop"] = vocab["term_str"].isin(STOPWORDS)

vocab = vocab[[
    "term_str",
    "n",
    "p",
    "i",
    "df",
    "idf",
    "dfidf",
    "porter_stem",
    "max_pos",
    "max_pos_group",
    "stop",
]]
vocab.to_csv("VOCAB.csv", index=True)

#### Top 20 vocab terms by dfidf

In [8]:
vocab.sort_values('dfidf', ascending=False).head(20)

,term_str,n,p,i,df,idf,dfidf,porter_stem,max_pos,max_pos_group,stop
term_id,,,,,,,,,,,
37,ill,214,0.004680,7.739260,48,1.437405,68.995455,ill,JJ,J,False
44,can,175,0.003827,8.029516,46,1.498806,68.945069,can,MD,V,True
58,out,138,0.003018,8.372203,51,1.349942,68.847066,out,RP,V,True
62,get,127,0.002777,8.492042,51,1.349942,68.847066,get,VB,V,False
45,youre,172,0.003762,8.054462,44,1.562936,68.769193,your,NN,N,False
49,not,157,0.003433,8.186106,52,1.321928,68.740261,not,RB,R,True
74,at,101,0.002209,8.822516,43,1.596103,68.632432,at,IN,I,True
59,down,138,0.003018,8.372203,43,1.596103,68.632432,down,RP,V,True
38,no,210,0.004593,7.766482,43,1.596103,68.632432,no,DT,D,True
